<a href="https://colab.research.google.com/github/isarandi/nlf/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
from PIL import Image

# Load model (once)
model = torch.jit.load('/content/nlf_l_multi_0.3.2.torchscript').cuda().eval()

pred_list = []
frame_ids = list(range(1523, 1529))

with torch.inference_mode():
    for fid in frame_ids:
        image_path = f'/content/IMG_{fid}.png'
        img = Image.open(image_path).convert('RGB')
        img_t = torchvision.transforms.functional.to_tensor(img).cuda()
        frame_batch = img_t.unsqueeze(0)

        pred = model.detect_smpl_batched(frame_batch, model_name='smplx')
        pred_list.append(pred)

print(f"Stored {len(pred_list)} predictions for frames {frame_ids[0]}..{frame_ids[-1]}")
print("Keys:", pred_list[0].keys())

Stored 6 predictions for frames 1523..1528
Keys: dict_keys(['boxes', 'pose', 'betas', 'trans', 'vertices3d', 'joints3d', 'vertices2d', 'joints2d', 'vertices3d_nonparam', 'joints3d_nonparam', 'vertices2d_nonparam', 'joints2d_nonparam', 'vertex_uncertainties', 'joint_uncertainties'])


In [ ]:

import torch

PELVIS = 0
HEAD = 15
LEFT_ANKLE = 7
RIGHT_ANKLE = 8

def to_Jx3(joints3d):
    # unwrap list outputs from NLF
    while isinstance(joints3d, list):
        if len(joints3d) == 0:
            raise ValueError("joints3d is an empty list")
        joints3d = joints3d[0]

    if not torch.is_tensor(joints3d):
        joints3d = torch.tensor(joints3d)

    # [1,J,3] -> [J,3]
    if joints3d.ndim == 3:
        joints3d = joints3d[0]

    return joints3d.detach().float().cpu()

ratios = []

for i, pred in enumerate(pred_list):
    joints = to_Jx3(pred["joints3d"])

    l_ankle = joints[LEFT_ANKLE]
    r_ankle = joints[RIGHT_ANKLE]
    pelvis = joints[PELVIS]
    head = joints[HEAD]

    ankle_dist = torch.norm(l_ankle - r_ankle)
    torso_dist = torch.norm(head - pelvis)

    ratio = (ankle_dist / torso_dist).item()
    ratios.append(ratio)

    print(f"Frame {1523+ i}: ankle_dist={ankle_dist.item():.4f}, head_pelvis_dist={torso_dist.item():.4f}, ratio={ratio:.6f}")

print("\nAll 3D ratios:", ratios)

Frame 1523: ankle_dist=145.5245, head_pelvis_dist=574.7585, ratio=0.253192
Frame 1524: ankle_dist=329.1918, head_pelvis_dist=596.2743, ratio=0.552081
Frame 1525: ankle_dist=427.5782, head_pelvis_dist=588.2610, ratio=0.726851
Frame 1526: ankle_dist=230.2738, head_pelvis_dist=586.6556, ratio=0.392520
Frame 1527: ankle_dist=126.0295, head_pelvis_dist=573.9400, ratio=0.219587
Frame 1528: ankle_dist=137.7962, head_pelvis_dist=577.6011, ratio=0.238566

All ratios: [0.25319236516952515, 0.5520812273025513, 0.7268510460853577, 0.39251965284347534, 0.2195865958929062, 0.23856638371944427]


In [ ]:
for i, pred in enumerate(pred_list):
    joints = to_Jx3(pred["joints3d"])

    #l_ankle = joints[LEFT_ANKLE]
    #r_ankle = joints[RIGHT_ANKLE]
    pelvis = joints[PELVIS]
    head = joints[HEAD]

    #print(l_ankle,r_ankle,pelvis,head)
    print(pelvis,head)

tensor([-271.8173,  -69.9097, 1805.7135]) tensor([-284.3524, -644.5287, 1803.9625])
tensor([ -21.1165,  -64.5038, 2046.3510]) tensor([ -66.8958, -658.4206, 2019.7029])
tensor([ 315.9840,  -70.7039, 2116.4187]) tensor([ 293.3201, -658.1766, 2096.0930])
tensor([ 508.4734,  -84.4933, 2237.9741]) tensor([ 478.7882, -670.2341, 2251.8054])
tensor([ 519.8934,  -99.8158, 2197.2031]) tensor([ 466.0524, -670.6152, 2170.8154])
tensor([ 428.1419, -102.5988, 2965.3535]) tensor([ 423.8600, -678.5470, 2921.8977])


In [ ]:
import torch

PELVIS = 0
HEAD = 15
LEFT_ANKLE = 7
RIGHT_ANKLE = 8

def to_Jx2(joints2d):
    # unwrap list outputs from NLF
    while isinstance(joints2d, list):
        if len(joints2d) == 0:
            raise ValueError("joints2d is an empty list")
        joints2d = joints2d[0]

    if not torch.is_tensor(joints2d):
        joints2d = torch.tensor(joints2d)

    # possible shapes: [1,J,2] -> [J,2]
    if joints2d.ndim == 3:
        joints2d = joints2d[0]

    return joints2d.detach().float().cpu()

ratios_2d = []

for i, pred in enumerate(pred_list):
    joints2d = to_Jx2(pred["joints2d"])

    l_ankle = joints2d[LEFT_ANKLE]
    r_ankle = joints2d[RIGHT_ANKLE]
    pelvis = joints2d[PELVIS]
    head = joints2d[HEAD]

    ankle_dist_2d = torch.norm(l_ankle - r_ankle)      # sqrt(dx^2 + dy^2)
    torso_dist_2d = torch.norm(head - pelvis)          # sqrt(dx^2 + dy^2)

    ratio_2d = (ankle_dist_2d / torso_dist_2d).item()
    ratios_2d.append(ratio_2d)

    print(f"Frame {1523 + i}: ankle_dist_2d={ankle_dist_2d.item():.4f}, head_pelvis_dist_2d={torso_dist_2d.item():.4f}, ratio_2d={ratio_2d:.6f}")

print("\nAll 2D ratios:", ratios_2d)

Frame 1505: ankle_dist_2d=252.5524, head_pelvis_dist_2d=941.3079, ratio_2d=0.268299
Frame 1506: ankle_dist_2d=201.9276, head_pelvis_dist_2d=764.5961, ratio_2d=0.264097
Frame 1507: ankle_dist_2d=161.5226, head_pelvis_dist_2d=603.4817, ratio_2d=0.267651
Frame 1508: ankle_dist_2d=141.5917, head_pelvis_dist_2d=475.0117, ratio_2d=0.298080
Frame 1509: ankle_dist_2d=172.2773, head_pelvis_dist_2d=640.5911, ratio_2d=0.268935
Frame 1510: ankle_dist_2d=231.2392, head_pelvis_dist_2d=813.2982, ratio_2d=0.284323
Frame 1511: ankle_dist_2d=215.5224, head_pelvis_dist_2d=950.1086, ratio_2d=0.226840
Frame 1512: ankle_dist_2d=85.6144, head_pelvis_dist_2d=515.7875, ratio_2d=0.165988

All 2D ratios: [0.2682994604110718, 0.26409706473350525, 0.26765114068984985, 0.2980804443359375, 0.26893484592437744, 0.28432273864746094, 0.22683973610401154, 0.1659877896308899]


In [ ]:
for k, v in pred.items():
    print(k, type(v))


boxes <class 'list'>
pose <class 'list'>
betas <class 'list'>
trans <class 'list'>
vertices3d <class 'list'>
joints3d <class 'list'>
vertices2d <class 'list'>
joints2d <class 'list'>
vertices3d_nonparam <class 'list'>
joints3d_nonparam <class 'list'>
vertices2d_nonparam <class 'list'>
joints2d_nonparam <class 'list'>
vertex_uncertainties <class 'list'>
joint_uncertainties <class 'list'>
